# Практика: логирование CatBoost в MLflow

Этот ноутбук показывает полный цикл:
1. Подключение к MLflow Tracking Server и MinIO
2. Подготовка данных
3. Обучение модели CatBoostClassifier
4. Логирование параметров, метрик и модели в MLflow
5. Загрузка модели из реестра и инференс

## 0) Установка зависимостей (если нужно)
Если запускаете впервые, раскомментируйте строку ниже и выполните ячейку.

In [ ]:
# !uv add catboost mlflow scikit-learn pandas python-dotenv

## 1) Импорты и настройки окружения

In [1]:
import os
import warnings

import pandas as pd
from dotenv import load_dotenv
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

import mlflow
import mlflow.catboost
from mlflow.models import infer_signature

warnings.filterwarnings("ignore")
load_dotenv()

True

In [2]:
# Для локального стенда из docker-compose
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "password")
raw_s3_endpoint = os.getenv("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
# `minio` резолвится только внутри docker-сети. Для локального ноутбука нужен localhost.
if "minio:9000" in raw_s3_endpoint:
    raw_s3_endpoint = "http://localhost:9000"
os.environ["MLFLOW_S3_ENDPOINT_URL"] = raw_s3_endpoint

# Если на сервере включена basic-auth, задайте логин/пароль
# os.environ["MLFLOW_TRACKING_USERNAME"] = os.getenv("MLFLOW_TRACKING_USERNAME", "admin")
# os.environ["MLFLOW_TRACKING_PASSWORD"] = os.getenv("MLFLOW_TRACKING_PASSWORD", "23wesdxc")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5050")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print("MLflow URI:", mlflow.get_tracking_uri())

MLflow URI: http://localhost:5050


In [3]:
# Быстрая проверка подключения
mlflow.search_experiments(max_results=3)

[<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1777816536833, experiment_id='2', last_update_time=1777816536833, lifecycle_stage='active', name='students-cnn-demo-proxy', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/', creation_time=1777816107810, experiment_id='1', last_update_time=1777816107810, lifecycle_stage='active', name='students-catboost-demo-proxy', tags={}>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1777816084738, experiment_id='0', last_update_time=1777816084738, lifecycle_stage='active', name='Default', tags={}>]

## 2) Подготовка датасета

In [4]:
df = pd.read_csv("data/apple_quality.csv")
df = df.dropna().copy()
df["target"] = (df["Quality"] == "good").astype(int)
df = df.drop(columns=["Quality", "A_id"])

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape, "test:", X_test.shape)

train: (3200, 7) test: (800, 7)


## 3) Настройка эксперимента и запуск run

In [5]:
from mlflow.tracking import MlflowClient

experiment_name = "students-catboost-demo-proxy"
artifact_location = "mlflow-artifacts:/"
client = MlflowClient()

exp = mlflow.get_experiment_by_name(experiment_name)
if exp is None:
    exp_id = client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_location
    )
    print("Created experiment:", exp_id)
else:
    exp_id = exp.experiment_id
    print("Using existing experiment:", exp_id, "artifact_location=", exp.artifact_location)

mlflow.set_experiment(experiment_name)

registered_model_name = "students_catboost_apple_quality"

params = {
    "iterations": 300,
    "depth": 6,
    "learning_rate": 0.05,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "verbose": False,
    "random_seed": 42
}


Using existing experiment: 1 artifact_location= mlflow-artifacts:/


In [6]:
with mlflow.start_run(experiment_id=exp_id):
    print("Artifact URI for this run:", mlflow.get_artifact_uri())
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    metrics = {
        "accuracy": float(accuracy_score(y_test, pred)),
        "f1": float(f1_score(y_test, pred)),
        "roc_auc": float(roc_auc_score(y_test, proba)),
        "log_loss": float(log_loss(y_test, proba)),
    }

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)

    signature = infer_signature(X_test, proba)

    model_info = mlflow.catboost.log_model(
        cb_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_test.head(5),
        registered_model_name=registered_model_name,
    )

    mlflow.log_input(
        mlflow.data.from_pandas(df, source="data/apple_quality.csv"),
        context="training",
    )

    # Маркируем текущую версию как PRD в реестре моделей
    client = MlflowClient()
    new_version = model_info.registered_model_version
    client.set_model_version_tag(registered_model_name, new_version, "env", "PRD")
    client.set_registered_model_alias(registered_model_name, "prd", new_version)

    run_id = mlflow.active_run().info.run_id
    print("Run ID:", run_id)
    print("Registered model version:", new_version)
    print("Alias 'prd' points to version:", new_version)
    print("Metrics:", metrics)


Artifact URI for this run: mlflow-artifacts:/df680aaa1b9b4a7f8f1fcb4bbaf5c2e9/artifacts


2026/05/03 17:01:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'students_catboost_apple_quality' already exists. Creating a new version of this model...
2026/05/03 17:01:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: students_catboost_apple_quality, version 4


Run ID: df680aaa1b9b4a7f8f1fcb4bbaf5c2e9
Registered model version: 4
Alias 'prd' points to version: 4
Metrics: {'accuracy': 0.89, 'f1': 0.891358024691358, 'roc_auc': 0.9547747173419835, 'log_loss': 0.2741558496925442}
🏃 View run zealous-cub-160 at: http://localhost:5050/#/experiments/1/runs/df680aaa1b9b4a7f8f1fcb4bbaf5c2e9
🧪 View experiment at: http://localhost:5050/#/experiments/1


Created version '4' of model 'students_catboost_apple_quality'.


### Если видите ошибку `experiment 0 ... deleted`
Это означает, что MLflow пытается писать в удаленный дефолтный эксперимент.
В этом ноутбуке run запускается с явным `experiment_id=exp_id`, поэтому просто перезапустите kernel и выполните ячейки сверху вниз.

## 4) Проверка результатов в MLflow

In [7]:
runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    order_by=["metrics.roc_auc DESC"],
)
runs_df[["run_id", "metrics.accuracy", "metrics.f1", "metrics.roc_auc", "artifact_uri"]].head()

,run_id,metrics.accuracy,metrics.f1,metrics.roc_auc,artifact_uri
0,df680aaa1b9b4a7f8f1fcb4bbaf5c2e9,0.89,0.891358,0.954775,mlflow-artifacts:/df680aaa1b9b4a7f8f1fcb4bbaf5...
1,169eb5a754b8452a8f3ab45377e0e65d,0.89,0.891358,0.954775,mlflow-artifacts:/169eb5a754b8452a8f3ab45377e0...
2,d67b894b6b23417893a15b5ec5018e08,0.89,0.891358,0.954775,mlflow-artifacts:/d67b894b6b23417893a15b5ec501...
3,e4134a7303a74f9fa930fea1fb35cc6b,0.89,0.891358,0.954775,mlflow-artifacts:/e4134a7303a74f9fa930fea1fb35...


In [8]:
# Показать запуски
mlflow.search_runs(experiment_names=[experiment_name])

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.log_loss,metrics.roc_auc,metrics.accuracy,metrics.f1,...,params.loss_function,params.learning_rate,params.depth,params.random_seed,params.verbose,tags.mlflow.log-model.history,tags.mlflow.runName,tags.mlflow.source.type,tags.mlflow.source.name,tags.mlflow.user
0,df680aaa1b9b4a7f8f1fcb4bbaf5c2e9,1,FINISHED,mlflow-artifacts:/df680aaa1b9b4a7f8f1fcb4bbaf5...,2026-05-03 14:01:33.907000+00:00,2026-05-03 14:01:35.791000+00:00,0.274156,0.954775,0.89,0.891358,...,Logloss,0.05,6,42,False,"[{""run_id"": ""df680aaa1b9b4a7f8f1fcb4bbaf5c2e9""...",zealous-cub-160,LOCAL,/Users/aweeu/Desktop/template-docker-mlflow-s3...,aweeu
1,169eb5a754b8452a8f3ab45377e0e65d,1,FINISHED,mlflow-artifacts:/169eb5a754b8452a8f3ab45377e0...,2026-05-03 13:53:46.318000+00:00,2026-05-03 13:53:47.744000+00:00,0.274156,0.954775,0.89,0.891358,...,Logloss,0.05,6,42,False,"[{""run_id"": ""169eb5a754b8452a8f3ab45377e0e65d""...",melodic-gull-355,LOCAL,/Users/aweeu/Desktop/template-docker-mlflow-s3...,admin
2,d67b894b6b23417893a15b5ec5018e08,1,FINISHED,mlflow-artifacts:/d67b894b6b23417893a15b5ec501...,2026-05-03 13:50:27.412000+00:00,2026-05-03 13:50:28.838000+00:00,0.274156,0.954775,0.89,0.891358,...,Logloss,0.05,6,42,False,"[{""run_id"": ""d67b894b6b23417893a15b5ec5018e08""...",catboost_baseline,LOCAL,/Users/aweeu/Desktop/template-docker-mlflow-s3...,admin
3,e4134a7303a74f9fa930fea1fb35cc6b,1,FINISHED,mlflow-artifacts:/e4134a7303a74f9fa930fea1fb35...,2026-05-03 13:48:28.459000+00:00,2026-05-03 13:48:30.594000+00:00,0.274156,0.954775,0.89,0.891358,...,Logloss,0.05,6,42,False,"[{""run_id"": ""e4134a7303a74f9fa930fea1fb35cc6b""...",catboost_baseline,LOCAL,/Users/aweeu/Desktop/template-docker-mlflow-s3...,admin


## 5) Загрузка модели из Model Registry по тегу PRD (alias `prd`)

In [16]:
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@prd")

sample_pred = loaded_model.predict(X_test.head(3))
sample_pred


array([1, 1, 0])